# Fine-tune the Industrial Automation Project Suggester model

Run this notebook on Google Colab (Runtime > Change runtime type > GPU) for a free GPU.
It clones your repo, generates the domain training data, LoRA fine-tunes `Qwen/Qwen2.5-0.5B-Instruct`,
and zips the resulting adapter so you can download it and commit it to `model/lora-adapter/` in your repo.

Takes roughly 10-15 minutes on a free Colab GPU.

In [ ]:
!pip -q install torch transformers peft datasets accelerate

In [ ]:
GITHUB_REPO_URL = "https://github.com/Abdelrahmanemad0/Industrial_Automation_project_suggester_chatbot.git"  # update if you renamed it
!git clone $GITHUB_REPO_URL repo
%cd repo

In [ ]:
!python scripts/generate_training_data.py
!python scripts/train_lora.py --epochs 3 --batch-size 4

In [ ]:
import shutil
shutil.make_archive('/content/lora-adapter', 'zip', 'model/lora-adapter')
print('Download /content/lora-adapter.zip from the Colab file browser,')
print('unzip it into model/lora-adapter/ in your repo, then commit + push.')

In [ ]:
# Optional: quick manual test of the fine-tuned adapter before you commit it.
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(base)
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base), 'model/lora-adapter')

messages = [
    {"role": "system", "content": "Reply with ONLY a JSON object: {\"rationale\": \"...\", \"buy_next\": \"...\"}."},
    {"role": "user", "content": "Student skill level: beginner\nSelected project: Ultrasonic Parking / Obstacle Distance Alarm\nHardware student already has: ['Arduino Uno', 'Ultrasonic Distance Sensor (HC-SR04)', 'Piezo Buzzer']\nHardware student is missing: none\n"},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors='pt')
out = model.generate(**inputs, max_new_tokens=120, do_sample=True, temperature=0.6)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))